In [ ]:
import numpy as np
import pandas as pd
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playerindex
from nba_api.stats.endpoints import leaguedashplayerbiostats

# **Data Processing Pipeline**

## Step 1: Datasets Merging, Alignment & Verification Pipeline

This step builds a unified shot tracking matrix by combining two highly different datasets. Because these files come from separate data vendors, we execute a multi-layered engineering pipeline to ensure a strict 1-to-1 match without data anomalies.

### The Component Datasets
1.  **`df1` (Official Play-by-Play Shot Logs):** Contains standard descriptive event fields (`ACTION_TYPE`, `SHOT_TYPE`, spatial coordinates `LOC_X` / `LOC_Y`) and the official outcome flag (`SHOT_MADE_FLAG`).
2.  **`df2` (Advanced Tracking Logs):** Contains high-fidelity player tracking attributes (`CLOSEST_DEFENDER`, `CLOSE_DEF_DIST`, physical tracking context like `TOUCH_TIME` and `DRIBBLES`), and the outcome string (`SHOT_RESULT`).

---

### Pipeline Execution Flow

The data engine processes the merging sequences in a strict, chronological flow:

#### 1. Data Loading (`pd.read_csv`)
Imports the official play-by-play logs (`df1`) and advanced tracking tables (`df2`) into the environment.

#### 2. Time Linearization (`MATCH_TIME`)
Normalizes game clocks across both datasets. Because one logs remaining period time and the other logs active possession clocks, we convert all timestamps into a continuous scalar: **total absolute seconds elapsed from the opening tip-off**.

#### 3. Type Sanitization & Guarding
Standardizes key columns by converting `GAME_ID` and `PLAYER_ID` to trimmed string formats (`.astype(str).str.strip()`), preventing whitespace mismatches from causing silent join drops.

#### 4. Core Logical Inner Join (Strict Constraints)
We pair data arrays using a multi-key validation check:
*   **Entity Linking:** Rows must share the exact same `GAME_ID` and `PLAYER_ID`.
*   **Outcome Identity Integration:** The shot results *must match perfectly* on both sides. A binary `1` (make) in `df1` is mapped against string values `made` in `df2`. If one dataset says a shot was made and the other says it was missed, the pair is immediately discarded.

#### 5. Smart Scoring Matrix Execution (Temporal Window & Penalties)
Because data vendors suffer from minor **sub-second clock drift**, we apply a **maximum 5-second difference threshold (`TIME_DIFF <= 5`)**. Within this 5-second cluster window, we run a semantic-physical scoring algorithm to eliminate false matches:

$$ Smart\_Rank\_Score = (5 - Time\_Diff) - Physical\_Penalty $$

*   **Baseline Proximity Score (`5 - TIME_DIFF`):** Closer timestamps score higher. A 0-second gap scores +5; a 3-second gap scores +2.
*   **Physical Rule Violations (`PHYSICAL_PENALTY`):** If tracking metrics violate the physical laws of the play-by-play description, a heavy **20-point penalty** is applied:
    *   *The Tip-In Violation:* Official description says `Tip-in` / `Putback`, but tracking claims `TOUCH_TIME > 1.5s` or `DRIBBLES > 0` $A \rightarrow B$ **Penalty + 20.0** (Demoted).
    *   *The Drive Violation:* Official description says `Driving` / `Pull-up`, but tracking records a catch-and-shoot with zero separation (`TOUCH_TIME <= 1.2s` and `DRIBBLES == 0`) $A \rightarrow B$ **Penalty + 20.0** (Demoted).

#### 6. Bidirectional Deduplication
The unified matrix is sorted by `SMART_RANK_SCORE` in descending order (pushing the most logically sound records to the top). We then execute a bidirectional `.drop_duplicates()` filter across both `MATCH_TIME_df1` and `MATCH_TIME_df2`. This safely enforces a mathematically sound **1-to-1 event mapping**.

In [3]:
df1 = pd.read_csv('data/nba_api_2014-15_shots.csv')
df2 = pd.read_csv('data/shot_logs(in).csv')

def get_total_elapsed_seconds_df1(row):
    period_duration = 720 if row['PERIOD'] <= 4 else 300
    secs_remaining = int(row['MINUTES_REMAINING']) * 60 + int(row['SECONDS_REMAINING'])
    secs_elapsed = period_duration - secs_remaining
    total_elapsed = sum([720 if p <= 4 else 300 for p in range(1, row['PERIOD'])]) + secs_elapsed
    return total_elapsed

def get_total_elapsed_seconds_df2(row):
    period_duration = 720 if row['PERIOD'] <= 4 else 300
    try:
        parts = str(row['GAME_CLOCK']).split(':')
        secs_remaining = int(parts[0]) * 60 + int(parts[1])
    except:
        secs_remaining = int(float(row['GAME_CLOCK']))
    secs_elapsed = period_duration - secs_remaining
    total_elapsed = sum([720 if p <= 4 else 300 for p in range(1, row['PERIOD'])]) + secs_elapsed
    return total_elapsed

df1['MATCH_TIME'] = df1.apply(get_total_elapsed_seconds_df1, axis=1)
df2['MATCH_TIME'] = df2.apply(get_total_elapsed_seconds_df2, axis=1)


df1['GAME_ID'] = df1['GAME_ID'].astype(str).str.strip()
df1['PLAYER_ID'] = df1['PLAYER_ID'].astype(str).str.strip()
df2['GAME_ID'] = df2['GAME_ID'].astype(str).str.strip()
df2['PLAYER_ID'] = df2['player_id'].astype(str).str.strip()


merged_df = pd.merge(df1, df2, on=['GAME_ID', 'PLAYER_ID'], suffixes=('_df1', '_df2'))

merged_df['TIME_DIFF'] = (merged_df['MATCH_TIME_df1'] - merged_df['MATCH_TIME_df2']).abs()
merged_df['DIST_DIFF'] = (merged_df['SHOT_DISTANCE'] - merged_df['SHOT_DIST']).abs()

df2_made_mapped = merged_df['SHOT_RESULT'].astype(str).str.lower().str.strip() == 'made'
merged_df['SHOT_MATCH_VALID'] = merged_df['SHOT_MADE_FLAG'].astype(int) == df2_made_mapped.astype(int)



# Smart Semantic-Physical Cross-Ranking Engine
action_lower = merged_df['ACTION_TYPE'].astype(str).str.lower()
merged_df['PHYSICAL_PENALTY'] = 0.0

# [Defense Line A]: If official log states a tip-in/putback, but tracking data 
# shows extended possession (>1.5s) or active dribbling, apply a heavy penalty (+20.0).
is_tip_action = action_lower.str.contains('tip|putback')
bad_tip_match = is_tip_action & ((merged_df['TOUCH_TIME'] > 1.5) | (merged_df['DRIBBLES'] > 0))
merged_df.loc[bad_tip_match, 'PHYSICAL_PENALTY'] += 20.0

# [Defense Line B]: If official log states an aggressive drive/on-ball play, but tracking 
# data shows zero dribbles and ultra-short possession (<=1.2s), apply a heavy penalty (+20.0).
drive_keywords = ['driving', 'pullup', 'step', 'turnaround', 'fadeaway', 'running']
is_drive_action = action_lower.str.contains('|'.join(drive_keywords))
bad_drive_match = is_drive_action & (merged_df['TOUCH_TIME'] <= 1.2) & (merged_df['DRIBBLES'] == 0)
merged_df.loc[bad_drive_match, 'PHYSICAL_PENALTY'] += 20.0

# Calculate the final Smart Ranking Score
merged_df['SMART_RANK_SCORE'] = (5 - merged_df['TIME_DIFF']) - merged_df['PHYSICAL_PENALTY']


valid_mask = (
    (merged_df['TIME_DIFF'] <= 5) & 
    (merged_df['SHOT_MATCH_VALID']) & 
    (merged_df['DIST_DIFF'] <= 3)
)
final_merged_df = merged_df[valid_mask].copy()

final_merged_df = final_merged_df.sort_values('SMART_RANK_SCORE', ascending=False)

# 1-to-1 event mapping
final_merged_df = final_merged_df.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID', 'MATCH_TIME_df1'], keep='first')
final_merged_df = final_merged_df.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID', 'MATCH_TIME_df2'], keep='first')

print(final_merged_df.head)


<bound method NDFrame.head of                  GRID_TYPE   GAME_ID  GAME_EVENT_ID PLAYER_ID  \
1105541  Shot Chart Detail  21400381            366    201196   
1539187  Shot Chart Detail  21400899            256      2590   
1105675  Shot Chart Detail  21400381            483    202326   
1315622  Shot Chart Detail  21400571            215    202683   
1167814  Shot Chart Detail  21400521            199      2564   
...                    ...       ...            ...       ...   
431722   Shot Chart Detail  21400861              8    201939   
943254   Shot Chart Detail  21400474             55    203496   
726784   Shot Chart Detail  21400737            341    203952   
451802   Shot Chart Detail  21400324            326      2772   
464654   Shot Chart Detail  21400512            214    202700   

                PLAYER_NAME     TEAM_ID               TEAM_NAME  PERIOD_df1  \
1105541      Ramon Sessions  1610612758        Sacramento Kings           3   
1539187         Mo Williams  16

## Step 2: Feature Engineering & Biographical Data Integration

We enrich matched shot logs with biographical metadata from the `nba_api` and engineer advanced features for our machine learning model:

*   **Biographical Alignment (`SHOOTER_` & `DEF_`):** Fetches player bio-stats for the 2014-15 season. Calculates league longevity (`EXP`) with a floor of 0, and separately maps traits (height, weight, age, experience) to both the shooter and the closest defender.
*   **Physical Deltas (`HEIGHT_DIFF`, `WEIGHT_DIFF`):** Calculates structural height and weight gaps between the shooter and the defender to capture mismatches.
*   **Temporal Compaction (`PERIOD_TIME_REMAINING`):** Flattens minutes and seconds into a single continuous scale of total remaining seconds per period.
*   **Action & Spatial Flags:** Converts text logs into binary indicators for specific shot tracking (`IS_CORNER_3`, `IS_RESTRICTED_AREA`, `IS_JUMP_SHOT`, `IS_DUNK_OR_LAYUP`) and strict tactical scenarios (`IS_CATCH_AND_SHOOT`).
*   **Volume & Micro-Momentum:** Maps in-game fatigue using cumulative attempt counts (`PLAYER_PRIOR_ATT`). Tracks moving hot-hand efficiency for players (`PLAYER_RFG_L5`) and teams (`TEAM_RFG_L10`) using a 1-shot lagged rolling window (imputed with a 0.45 baseline).
*   **Advanced Interaction Features:** Engineers combined variables to capture high-level dynamics, including shooting streaks vs. distance (`HOT_HAND_X_...`), fatigue vs. time (`FATIGUE_INDEX`), experience under pressure (`VETERAN_CLUTCH_SCORE`), and team momentum (`TEAM_MOMENTUM_X_MARGIN`).


In [7]:
bio_stats = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season='2014-15')
all_players_data = bio_stats.get_data_frames()[0]

player_detail = all_players_data[['PLAYER_ID', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'AGE', 'DRAFT_YEAR']].copy()
player_detail.columns = ['PLAYER_ID', 'HEIGHT_INCHES', 'WEIGHT', 'SEASON_AGE', 'DRAFT_YEAR']
player_detail = player_detail.dropna(subset=['HEIGHT_INCHES', 'WEIGHT']).copy()

player_detail['DRAFT_YEAR'] = pd.to_numeric(player_detail['DRAFT_YEAR'], errors='coerce').fillna(2014)
player_detail['EXP'] = 2014 - player_detail['DRAFT_YEAR']
player_detail.loc[player_detail['EXP'] < 0, 'EXP'] = 0

#all_players_data = playerindex.PlayerIndex(historical_nullable=1).get_data_frames()[0]
#all_players_data = playerindex.PlayerIndex(season="2014-15").get_data_frames()[0]

#player_detail = all_players_data[['PERSON_ID', 'HEIGHT', 'WEIGHT']]
#player_detail.columns = ['PLAYER_ID', 'HEIGHT', 'WEIGHT']
#player_detail = player_detail.dropna(subset=['HEIGHT', 'WEIGHT'])

#split_height = player_detail['HEIGHT'].str.split('-', expand=True).astype(int)
#player_detail['HEIGHT_INCHES'] = (split_height[0]*12) + split_height[1]

df_feature = final_merged_df.copy()
player_detail['PLAYER_ID'] = player_detail['PLAYER_ID'].astype(str).str.strip()
df_feature['PLAYER_ID'] = df_feature['PLAYER_ID'].astype(str).str.strip()
df_feature['CLOSEST_DEFENDER_PLAYER_ID'] = df_feature['CLOSEST_DEFENDER_PLAYER_ID'].astype(str).str.strip()

shooter_info = (
    player_detail[['PLAYER_ID', 'HEIGHT_INCHES', 'WEIGHT', 'SEASON_AGE', 'EXP']]
    .rename(columns={
        'HEIGHT_INCHES': 'SHOOTER_HEIGHT', 
        'WEIGHT': 'SHOOTER_WEIGHT'
    })
)
df_feature = pd.merge(df_feature, shooter_info, on='PLAYER_ID', how='left')

defender_info = (
    player_detail[['PLAYER_ID', 'WEIGHT', 'HEIGHT_INCHES']]
    .rename(columns={
        'PLAYER_ID': 'CLOSEST_DEFENDER_PLAYER_ID',
        'WEIGHT': 'DEF_WEIGHT',
        'HEIGHT_INCHES': 'DEF_HEIGHT_INCHES'
    })
)
df_feature = pd.merge(df_feature, defender_info, on='CLOSEST_DEFENDER_PLAYER_ID', how='left')


for col in ['SHOOTER_HEIGHT', 'DEF_HEIGHT_INCHES', 'SHOOTER_WEIGHT', 'DEF_WEIGHT', 'SEASON_AGE']:
    df_feature[col] = pd.to_numeric(df_feature[col], errors='coerce')

df_feature['HEIGHT_DIFF'] = df_feature['SHOOTER_HEIGHT'] - df_feature['DEF_HEIGHT_INCHES']
df_feature['WEIGHT_DIFF'] = df_feature['SHOOTER_WEIGHT'] - df_feature['DEF_WEIGHT']
df_feature['PERIOD_TIME_REMAINING'] = (df_feature['MINUTES_REMAINING'].astype(int)*60) + df_feature['SECONDS_REMAINING'].astype(int)


df_feature['GAME_DATE'] = pd.to_datetime(df_feature['GAME_DATE'])
df_feature = df_feature.sort_values(
    by=['GAME_DATE', 'GAME_ID', 'PERIOD_df1', 'PERIOD_TIME_REMAINING'], 
    ascending=[True, True, True, False]
).reset_index(drop=True)


for col in ['SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'ACTION_TYPE', 'SHOT_TYPE']:
    df_feature[col] = df_feature[col].astype(str).str.strip()

df_feature['IS_CORNER_3'] = df_feature['SHOT_ZONE_BASIC'].str.contains('Corner 3', case=False, na=False).astype(int)
df_feature['IS_RESTRICTED_AREA'] = (df_feature['SHOT_ZONE_BASIC']=='Restricted Area').astype(int)
df_feature['IS_PAINT'] = (df_feature['SHOT_ZONE_BASIC']=='In The Paint (Non-RA)').astype(int)
df_feature['IS_MID_RANGE'] = (df_feature['SHOT_ZONE_BASIC']=='Mid-Range').astype(int)
df_feature['IS_BREAK_3'] = (df_feature['SHOT_ZONE_BASIC']=='Above the Break 3').astype(int)

df_feature['IS_JUMP_SHOT'] = df_feature['ACTION_TYPE'].str.contains('Jump Shot', case=False, na=False).astype(int)
df_feature['IS_DUNK_OR_LAYUP'] = df_feature['ACTION_TYPE'].str.contains('Dunk|Layup', case=False, na=False).astype(int)
df_feature['IS_CATCH_AND_SHOOT'] = ((df_feature['DRIBBLES']==0)&(df_feature['TOUCH_TIME']<=2)&(df_feature['IS_JUMP_SHOT']==1)).astype(int)


df_feature['PLAYER_PRIOR_ATT'] = df_feature.groupby(['GAME_ID', 'PLAYER_ID']).cumcount()

#Rolling Field Goal Last x of player and team
df_feature['PLAYER_RFG_L5'] = (
    df_feature.groupby('PLAYER_ID')['SHOT_MADE_FLAG']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
).fillna(0.45) 

df_feature['TEAM_RFG_L10'] = (
    df_feature.groupby('TEAM_ID')['SHOT_MADE_FLAG']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
).fillna(0.45)


#New Interact Feature
df_feature['FATIGUE_INDEX'] = df_feature['PLAYER_PRIOR_ATT'] * (df_feature['PERIOD_df1'] + (1 - df_feature['PERIOD_TIME_REMAINING']/720))
df_feature['TEAM_MOMENTUM_X_MARGIN'] = df_feature['TEAM_RFG_L10'] * df_feature['FINAL_MARGIN']

df_feature['SPATIAL_PRESSURE_IDX'] = df_feature['SHOT_DIST'] / (df_feature['CLOSE_DEF_DIST'] + 0.1)
df_feature['PURE_CATCH_AND_SHOOT_GAP'] = (df_feature['DRIBBLES'] == 0).astype(int) * df_feature['CLOSE_DEF_DIST']
df_feature['HOT_HAND_X_JUMP_SHOT'] = df_feature['PLAYER_RFG_L5'] * df_feature['IS_JUMP_SHOT']


print(df_feature.head)

<bound method NDFrame.head of                 GRID_TYPE   GAME_ID  GAME_EVENT_ID PLAYER_ID  \
0       Shot Chart Detail  21400001              2    203076   
1       Shot Chart Detail  21400001              4    202696   
2       Shot Chart Detail  21400001              7    203076   
3       Shot Chart Detail  21400001              9    203901   
4       Shot Chart Detail  21400001             26    202696   
...                   ...       ...            ...       ...   
102238  Shot Chart Detail  21400908            601    101108   
102239  Shot Chart Detail  21400908            603    200746   
102240  Shot Chart Detail  21400908            604    200755   
102241  Shot Chart Detail  21400908            606    201587   
102242  Shot Chart Detail  21400908            613    101108   

              PLAYER_NAME     TEAM_ID               TEAM_NAME  PERIOD_df1  \
0           Anthony Davis  1610612740    New Orleans Pelicans           1   
1          Nikola Vučević  1610612753          

## Step 3: Matrix Pruning & Quality Assurance

We filter the dataset to target features, clean naming conventions, and run a data quality audit to eliminate corrupted logs before modeling.

* **Dimensionality Reduction**: Removed redundant diagnostic tracking features, keeping only high-signal markers.
* **Structural Refactoring**: Renamed technical merge labels (e.g., `PERIOD_df1`, `MATCH_TIME_df1`) to clean variables (`PERIOD`, `MATCH_TIME`).
* **Quality Assurance Audits**:
  * **Possession Time Audit**: Filtered out **3,358 records with `TOUCH_TIME == 0`** and **312 records with `TOUCH_TIME < 0`** to eliminate these physically impossible tracking anomalies.
  * **Missing Values**: Executed `.dropna()` to clear remaining missing values (NaN) to protect model stability.
* **Final Preprocessed Volume**: The final cleaned dataset contains **95,892 clean shot rows** saved directly to `data/shotlog_2014-15_preprocessed.csv`.


In [8]:
columns_to_keep = [
    # IDs & Names
    'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME',
    
    # Game Context & Time
    'GAME_DATE', 'HTM', 'VTM', 'LOCATION', 'MATCH_TIME_df1', 'PERIOD_df1', 
    'PERIOD_TIME_REMAINING', 'FINAL_MARGIN',
    
    # Shot Details
    'GRID_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 
    'SHOT_ZONE_RANGE', 'SHOT_DIST', 'LOC_X', 'LOC_Y', 
    'SHOT_CLOCK', 'DRIBBLES', 'TOUCH_TIME',
    
    # Outcomes
    'SHOT_MADE_FLAG',
    
    # Defender Data
    'CLOSEST_DEFENDER', 'CLOSEST_DEFENDER_PLAYER_ID', 'CLOSE_DEF_DIST',
    
    # Player Physicals & Mismatches
    'SHOOTER_HEIGHT', 'SHOOTER_WEIGHT', 'DEF_HEIGHT_INCHES', 'DEF_WEIGHT', 'HEIGHT_DIFF', 'WEIGHT_DIFF',
    
    # Dynamic Form Features
    'SEASON_AGE', 'EXP', 'PLAYER_PRIOR_ATT', 'PLAYER_RFG_L5', 'TEAM_RFG_L10',

    # Engineered Tactical Features
    'IS_CORNER_3', 'IS_RESTRICTED_AREA', 'IS_PAINT', 'IS_MID_RANGE', 'IS_BREAK_3',
    'IS_JUMP_SHOT', 'IS_DUNK_OR_LAYUP', 'IS_CATCH_AND_SHOOT',

    # Advanced Interaction Features
    'FATIGUE_INDEX', 'TEAM_MOMENTUM_X_MARGIN',

    'SPATIAL_PRESSURE_IDX', 'PURE_CATCH_AND_SHOOT_GAP', 'HOT_HAND_X_JUMP_SHOT'
]   

df_cleaned = df_feature[columns_to_keep].copy()
df_cleaned = df_cleaned[df_cleaned['TOUCH_TIME'] > 0].copy()

rename_dict = {
    'MATCH_TIME_df1': 'MATCH_TIME',
    'PERIOD_df1': 'PERIOD',
}
df_cleaned = df_cleaned.rename(columns=rename_dict)
df_cleaned.dropna(inplace=True)

df_cleaned.to_csv('data/shotlog_2014-15_preprocessed.csv', index=False)

print(len(df_cleaned))
df_cleaned.head()

95729


,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,GAME_DATE,HTM,VTM,LOCATION,...,IS_MID_RANGE,IS_BREAK_3,IS_JUMP_SHOT,IS_DUNK_OR_LAYUP,IS_CATCH_AND_SHOOT,FATIGUE_INDEX,TEAM_MOMENTUM_X_MARGIN,SPATIAL_PRESSURE_IDX,PURE_CATCH_AND_SHOOT_GAP,HOT_HAND_X_JUMP_SHOT
0,21400001,2,203076,Anthony Davis,1610612740,New Orleans Pelicans,1970-01-01 00:00:00.020141028,NOP,ORL,H,...,1,0,1,0,1,0.000000,7.65,4.106383,4.6,0.45
1,21400001,4,202696,Nikola Vučević,1610612753,Orlando Magic,1970-01-01 00:00:00.020141028,NOP,ORL,A,...,1,0,0,0,0,0.000000,-7.65,3.015152,6.5,0.00
2,21400001,7,203076,Anthony Davis,1610612740,New Orleans Pelicans,1970-01-01 00:00:00.020141028,NOP,ORL,H,...,1,0,1,0,1,1.075000,0.00,1.743590,11.6,0.00
3,21400001,9,203901,Elfrid Payton,1610612753,Orlando Magic,1970-01-01 00:00:00.020141028,NOP,ORL,A,...,0,0,0,1,0,0.000000,-17.00,1.047619,0.0,0.00
4,21400001,26,202696,Nikola Vučević,1610612753,Orlando Magic,1970-01-01 00:00:00.020141028,NOP,ORL,A,...,0,0,0,0,0,1.140278,-8.50,3.600000,0.0,0.00
